# Data Skewness in PySpark: Detection and Correct Mitigation

This notebook demonstrates how to identify and mitigate **data skew** in Spark using a synthetic 10-million-row DataFrame. It covers:

- detecting skew,
- validating a **baseline skewed aggregation**,
- fixing skew with **salting**,
- a **targeted salting** variant that salts only the hot key,
- and validating that optimized results are identical to the baseline.

> Note: **AQE skew handling helps mainly for joins**, not for skewed aggregations such as `groupBy(skew_key)`.


In [0]:
from pyspark.sql import functions as F
from pyspark.storagelevel import StorageLevel
import time

# --- Generate 10M records with a deliberately skewed key distribution ---
# 90% of records get key='A', the remaining 10% are spread across other keys.

num_records = 10_000_000
num_partitions = 20

# Distributed generation, no driver bottleneck
df = spark.range(0, num_records, step=1, numPartitions=num_partitions)

# Create skewed key: 90% -> 'A', rest spread across key_2..key_10
df = df.withColumn(
    "skew_key",
    F.when(F.col("id") < F.lit(num_records * 0.9), F.lit("A"))
     .otherwise(F.concat(F.lit("key_"), (F.col("id") % 9 + 2).cast("string")))
)

# Add a numeric value column for aggregation
df = df.withColumn("value", F.rand(seed=42) * 1000)

# Intentionally repartition by the skewed key to make the imbalance visible
df = df.repartition(num_partitions, "skew_key").persist(StorageLevel.MEMORY_AND_DISK)
_ = df.count()

print(f"Total records: {num_records:,}")
print(f"Configured partitions: {num_partitions}")
display(df.limit(10))


## 1. Detect the Skew

Before fixing the problem, measure how records are distributed across keys and partitions.
A key that dominates the volume (here `A` at ~90%) creates a straggler during shuffle-heavy operations such as aggregation.


In [0]:
# --- Show key distribution to confirm the skew ---
key_distribution = df.groupBy("skew_key").agg(
    F.count("*").alias("record_count"),
    (F.count("*") / num_records * 100).alias("percentage")
).orderBy(F.col("record_count").desc())

display(key_distribution)

In [0]:
# --- Show per-partition record counts to visualise the imbalance ---
# After repartition("skew_key"), all 'A' records land in one partition -> massive straggler
partition_sizes = df.select(
    F.spark_partition_id().alias("partition_id")
).groupBy("partition_id").agg(
    F.count("*").alias("record_count")
).orderBy("partition_id")

display(partition_sizes)

## 2. Baseline: Aggregation on Skewed Data

This is a standard `groupBy("skew_key")` without skew mitigation.
Because one key dominates the data, one reducer has to process most of the records for that key.


In [0]:
# --- Baseline aggregation: skewed groupBy ---
start = time.time()

result_baseline = df.groupBy("skew_key").agg(
    F.sum("value").alias("total_value"),
    F.avg("value").alias("avg_value"),
    F.count("*").alias("record_count")
).orderBy("skew_key")

baseline_count = result_baseline.count()
elapsed_baseline = time.time() - start

print(f"Baseline groupBy (skewed): {elapsed_baseline:.2f}s -> {baseline_count} groups")
display(result_baseline)


## 3. Solution 1: Full Salting

**Salting** breaks a hot key into `N` sub-keys. Aggregation is done in two stages:

1. Add `salt`, then aggregate by `(skew_key, salt)`.
2. Drop salt and re-aggregate by `skew_key`.

This distributes work for the hot key across multiple reducers.

Implementation note:
- use `pmod(hash(...), N)` instead of `%`, because `%` may produce negative values in Spark.


In [0]:
# --- Full salting solution ---
SALT_BUCKETS = 10

start = time.time()

df_salted = df.withColumn(
    "salt",
    F.pmod(F.hash("id"), F.lit(SALT_BUCKETS)).cast("int")
)

stage1 = df_salted.groupBy("skew_key", "salt").agg(
    F.sum("value").alias("sum_value"),
    F.count("*").alias("record_count")
)

result_salted = stage1.groupBy("skew_key").agg(
    F.sum("sum_value").alias("total_value"),
    (F.sum("sum_value") / F.sum("record_count")).alias("avg_value"),
    F.sum("record_count").alias("record_count")
).orderBy("skew_key")

salted_count = result_salted.count()
elapsed_salted = time.time() - start

print(f"Salted groupBy: {elapsed_salted:.2f}s -> {salted_count} groups")
display(result_salted)


## 4. Solution 2: Targeted Salting for Hot Key Only

In production, salting every key is often unnecessary.
A better approach is to salt only the detected hot key (`A` here), while leaving other keys unchanged.

This reduces overhead and keeps the optimization focused on the real skew source.


In [0]:
start = time.time()

HOT_KEY = "A"
SUB_GROUPS = 10

# Salt only the hot key; keep all other keys in a single subgroup 0
df_targeted = df.withColumn(
    "sub_group",
    F.when(
        F.col("skew_key") == F.lit(HOT_KEY),
        F.pmod(F.hash("id"), F.lit(SUB_GROUPS)).cast("int")
    ).otherwise(F.lit(0))
)

# Optional: repartition by the expanded key to visualize a more even layout
df_targeted = df_targeted.repartition(num_partitions, "skew_key", "sub_group").persist(StorageLevel.MEMORY_AND_DISK)
_ = df_targeted.count()

balanced_partition_sizes = df_targeted.select(
    F.spark_partition_id().alias("partition_id")
).groupBy("partition_id").agg(
    F.count("*").alias("record_count")
).orderBy("partition_id")

print("Partition sizes after targeted salting:")
display(balanced_partition_sizes)


In [0]:
# --- Aggregation using targeted salting ---
result_targeted = df_targeted.groupBy("skew_key", "sub_group").agg(
    F.sum("value").alias("sum_value"),
    F.count("*").alias("record_count")
).groupBy("skew_key").agg(
    F.sum("sum_value").alias("total_value"),
    (F.sum("sum_value") / F.sum("record_count")).alias("avg_value"),
    F.sum("record_count").alias("record_count")
).orderBy("skew_key")

targeted_count = result_targeted.count()
elapsed_targeted = time.time() - start

print(f"Targeted salting groupBy: {elapsed_targeted:.2f}s -> {targeted_count} groups")
display(result_targeted)


## 5. What About `repartition(50)`?

A plain `df.repartition(50)` is **not a real fix** for skewed aggregation by key.

Why?
- `groupBy("skew_key")` triggers a new shuffle by `skew_key`.
- all rows for the hot key still have to meet in the same reduce path,
- so the bottleneck remains.

Increasing partition count may help some upstream steps, but it does **not** solve the core skew problem for hot-key aggregation.


In [0]:
# --- Demonstrate why simple repartition is not a skew fix ---
start = time.time()

df_repartitioned = df.repartition(50)

repartition_sizes = df_repartitioned.select(
    F.spark_partition_id().alias("partition_id")
).groupBy("partition_id").agg(
    F.count("*").alias("record_count")
).orderBy("partition_id")

print("Partition distribution after repartition(50):")
display(repartition_sizes)

result_repartition = df_repartitioned.groupBy("skew_key").agg(
    F.sum("value").alias("total_value"),
    (F.sum("value") / F.count("*")).alias("avg_value"),
    F.count("*").alias("record_count")
).orderBy("skew_key")

repartition_count = result_repartition.count()
elapsed_repartition = time.time() - start

print(f"Plain repartition(50) + groupBy: {elapsed_repartition:.2f}s -> {repartition_count} groups")
display(result_repartition)


## 6. Validation and Summary

Below we validate that both salting-based approaches return the **same business result** as the baseline.
This is essential: performance optimization must not change correctness.


In [0]:
# --- Validation and summary ---
validation_salted = result_baseline.exceptAll(result_salted).count() + result_salted.exceptAll(result_baseline).count()
validation_targeted = result_baseline.exceptAll(result_targeted).count() + result_targeted.exceptAll(result_baseline).count()
validation_repartition = result_baseline.exceptAll(result_repartition).count() + result_repartition.exceptAll(result_baseline).count()

summary_data = [
    ("Baseline (skewed groupBy)", round(elapsed_baseline, 2), baseline_count, "reference"),
    ("Full salting", round(elapsed_salted, 2), salted_count, "OK" if validation_salted == 0 else f"DIFF={validation_salted}"),
    ("Targeted salting", round(elapsed_targeted, 2), targeted_count, "OK" if validation_targeted == 0 else f"DIFF={validation_targeted}"),
    ("Plain repartition(50)", round(elapsed_repartition, 2), repartition_count, "OK" if validation_repartition == 0 else f"DIFF={validation_repartition}"),
]

summary_df = spark.createDataFrame(
    summary_data,
    schema=["approach", "elapsed_seconds", "group_count", "validation_status"]
)

display(summary_df)

print("--- Key Takeaways ---")
print("1. Skewed data creates stragglers because one hot key dominates the shuffle workload.")
print("2. Full salting is a correct mitigation for skewed aggregations.")
print("3. Targeted salting is usually better operationally because it optimizes only the hot key.")
print("4. Plain repartitioning alone does not solve hot-key aggregation skew.")
print("5. AQE helps mainly with skewed joins, not with groupBy skew itself.")

print("--- When to Use Each Method ---")
print("- Full salting: when skew is severe and multiple keys may be problematic.")
print("- Targeted salting: when one or a few hot keys are known or detected.")
print("- Plain repartition: for generic reshaping/parallelism, not as the core skew fix.")
print("- AQE: especially for joins, where Spark can split skewed partitions automatically.")
